# Mechanistic Interpretability: From Black Boxes to Sparse Features (SAEs)

This notebook is a hands-on tutorial for a mechanistic interpretability bootcamp.  
We follow a simple story arc:

1. **The Problem**: Why LLMs feel like black boxes; why single neurons are unreliable due to superposition.
2. **Sparse Autoencoders (SAEs) as a Solution**: Turning dense activations into a sparse, interpretable feature dictionary.
3. **Feature Steering Demo**: Nudging the model by clamping or amplifying a feature.
4. **The Dark Matter**: Quantifying what our features **do not** explain (reconstruction error + behavior gap).

> Practical note: This tutorial uses **Gemma 2B** via TransformerLens and a **pretrained SAE** from the Gemma-Scope / SAELens ecosystem.  
> If you run this in Colab, a GPU is recommended.

---

## References (good starting points)

- TransformerLens (mechanistic interpretability tooling):  
  https://github.com/TransformerLensOrg/TransformerLens  
- Sparse Autoencoders for interpretability (community tooling + docs):  
  SAELens: https://github.com/jbloomAus/SAELens  
- Gemma Scope (pretrained SAEs for Gemma layers, plus Neuronpedia integration):  
  https://deepmind.google/models/gemma/gemma-scope/   
- Neuronpedia (browse features interactively):  
  https://neuronpedia.org/  
- Classic “superposition” framing (neurons as mixed features):  
  Anthropic, “Toy Models of Superposition” (2022): https://transformer-circuits.pub/2022/toy_model/index.html

---

## Learning outcomes

By the end, you should be able to:

- Explain **superposition** in one sentence and why it breaks "neuron = concept".
- Load an LLM and a pretrained **SAE**, and run: `activations → encoder → sparse features → decoder → reconstruction`.
- Identify a **human-legible feature** using its top-activating examples.
- Perform **feature steering** with a single line hook.
- Compute a simple **dark matter score**: how much activation energy and next-token behavior remain unexplained.


> There is a trade-off between model and SAE choices:
* Best-matching SAE dictionary: typically 2b-pt (SAEs trained on base/PT)

* Most compliant generation behavior: typically 2b-it

In [ ]:
import os; os.environ.setdefault('HF_HUB_DISABLE_XET', '1')  # avoid full-repo downloads
# ============================================================
# MODEL SELECTION — change this to switch the entire tutorial
# ============================================================

MODEL_FAMILY = "gpt2"  # "gpt2" | "gemma"

# That's it. Everything else is derived automatically below.

## 0.1 Imports and device

We will use:

- **TransformerLens** to load the model and cache activations.
- **sae-lens** to load a pretrained SAE and run encoder/decoder.

If you cannot download Gemma weights due to access restrictions, you will need to authenticate with HuggingFace.
In Colab, one common pattern is:

```python
from huggingface_hub import login
login()  # then paste your token
```
or in terminal:
```bash
huggingface-cli login  # then paste your token
```

(We do not run `login()` automatically; you should do so only if needed.)


In [ ]:
if MODEL_FAMILY == "gemma":
    from huggingface_hub import login

    login()
else:
    print("GPT-2 is publicly available — no login needed.")

GPT-2 is publicly available — no login needed.


In [ ]:
import random
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from sae_lens import SAE
from transformer_lens import HookedTransformer

In [ ]:
# This configuration dictionary centralizes the parameters for both model options.
# You do not need to understand every field yet.
# We will explain each one when it first becomes relevant.
CONFIGS = {
    "gpt2": {
        "model_name": "gpt2",
        "layer": 8,
        "hook_point": "blocks.8.hook_resid_pre",
        "sae_release": "gpt2-small-res-jb",
        "sae_id": "blocks.8.hook_resid_pre",
        "np_model_id": "gpt2-small",
        "np_sae_id": "8-res-jb",
    },
    "gemma": {
        "model_name": "google/gemma-2-2b",
        "layer": 20,
        "hook_point": "blocks.20.hook_mlp_out",
        "sae_release": "gemma-scope-2b-pt-mlp-canonical",
        "sae_id": "layer_20/width_16k/canonical",
        "np_model_id": "gemma-2-2b",
        "np_sae_id": "20-gemmascope-mlp-16k",
    },
}

cfg = CONFIGS[MODEL_FAMILY]
print(f"Config loaded for: {MODEL_FAMILY} Successfully!")

Config loaded for: gpt2 Successfully!


> ⚠️Warning: These are two different streams. The SAE gpt2-small-res-jb is trained on the residual stream, and the hook_point must also correspond to hook_resid_pre. Otherwise, feeding the activations from the MLP-out to the SAE on the residual stream will be completely meaningless.

In [ ]:
SEED = 0
DEVICE = "cuda"  # fallback handled later
DTYPE = "bfloat16"  # consider "bfloat16" on newer GPUs, else float16/float32
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

Device: cuda


# 1) The Problem: Why are LLMs "black boxes"?

A modern LLM is a pile of matrix multiplications with **billions of parameters**.  
Even if you can inspect weights, *interpretation* is hard because:

- Computation is distributed across layers, heads, and MLPs.
- Internal representations are **high-dimensional** (hundreds to thousands of dims).
- Individual neurons are often **polysemantic** (respond to multiple unrelated things).

### Superposition (one neuron = many concepts)

**Superposition** is the idea that neural nets often represent many “features” in a compressed way:
multiple features are **superimposed** in the same neurons/directions.

One sentence version:
> A neuron’s activation is usually a messy mixture of many features, so “neuron = concept” is a bad default.

For example, a dimension might simultaneously encode "animal," "vehicle," and "emotion." When "happy dog" is input, that dimension is activated; when "sad car" is input, that dimension is also activated.

We'll make that concrete: first we will look at dense activations; then we will introduce a dictionary of sparse features learned by a Sparse Autoencoder (SAE).


## 1.1 Quick “black box” sanity: a model is huge

We will load the selected model and print a few configuration stats.
This is just a quick sanity check before we move to sparse autoencoders.

In [ ]:
# ============================================================
# 1.1 Load model (TransformerLens)
# ============================================================

model = HookedTransformer.from_pretrained(cfg["model_name"], device=device)

print("Layers:", model.cfg.n_layers)
print("Heads:", model.cfg.n_heads)
print("d_model:", model.cfg.d_model)
print("Vocab:", model.cfg.d_vocab)

Loaded pretrained model gpt2 into HookedTransformer
Layers: 12
Heads: 12
d_model: 768
Vocab: 50257


# 2) Sparse Autoencoders (SAEs) as the Solution

## 2.1 What an SAE is doing (conceptually)

At some layer, the model produces an activation vector $$x \in \mathbb{R}^{d}$$.  
An SAE learns:

- **Encoder**: $$z = f(W_{enc} x + b)$$ with a sparsity constraint (most entries of \(z\) are 0).

- **Decoder**: $$\hat{x} = W_{dec} z + b'$$
- Objective: reconstruction fidelity + sparsity of \(z\).

**What is $f$?** It depends on the SAE variant:

| SAE type | $f$ | Sparsity mechanism |
|---|---|---|
| **Standard / vanilla SAE** | ReLU$(x) = \max(0, x)$ | Negative values → exactly zero |
| **TopK SAE** | Keep top-$k$ entries, zero the rest | Hard sparsity budget |
| **JumpReLU SAE** | ReLU with a learned per-feature threshold $\theta$: $f(x) = x \cdot \mathbf{1}[x > \theta]$ | Threshold trained jointly; more adaptive than ReLU |

**Gemma Scope uses JumpReLU**. This means each feature has a learned firing threshold —
a feature only activates if its pre-activation exceeds that threshold.
The advantage: JumpReLU SAEs achieve a better reconstruction–sparsity tradeoff than vanilla ReLU SAEs.

For our tutorial purposes the practical difference is small:
`sae.encode(x)` returns the sparse code regardless of which variant is used.

**Interpretability hope**:
- Each coordinate of \(z\) (a *latent feature*) corresponds to a more *coherent* concept than a raw neuron dimension.

We will:
1. Grab activations from one layer (MLP output is a common choice).
2. Run them through a pretrained SAE.
3. Inspect which features fire the most on some prompts.
4. Use Neuronpedia dashboards to visualize features (when available).

> We intentionally keep the dataset small: this is a tutorial, not a large-scale interpretability study.


## 2.2 Choose a hook point

A practical SAE tutorial needs a stable “tap point” in the model.  
A common choice is the **MLP output** at some layer, e.g.:

- `blocks.{L}.hook_mlp_out`

We'll use a middle-ish layer. For Gemma 2B, layer ~20 is often used in examples, but any layer with an available pretrained SAE works.

For our chosen model, we use layer `cfg['layer']`.


In [ ]:
LAYER = cfg["layer"]
HOOK_POINT = cfg["hook_point"]
print("Model:", cfg["model_name"])
print("Hook point:", HOOK_POINT)

Model: gpt2
Hook point: blocks.8.hook_resid_pre


> **Why different hook points for different models?**
>
> - **GPT-2 SAE** (`blocks.8.hook_resid_pre`): trained on the **residual stream** —
>   the main information highway that accumulates representations across all layers.
>   Residual-stream SAEs capture a richer mix of all computations up to that point.
>
> - **Gemma SAE** (`blocks.20.hook_mlp_out`): trained on the **MLP output** —
>   the output of the feed-forward network at layer 20, before it is added back to
>   the residual stream.  MLP-output SAEs capture features that the MLP layer specifically
>   "writes" in this step.
>
> **Critical constraint**: the hook point used during *inference* must exactly match
> the hook point used during *SAE training*. The SAE encoder was fit to the statistical
> distribution of activations at a specific location. Feeding activations from a different
> location (e.g., MLP-out to a residual-stream SAE) would produce meaningless sparse codes,
> because the input distribution is completely different.
>
> The `CONFIGS` dictionary enforces this pairing: `hook_point` and `sae_id` are always
> set together to a consistent (model, layer, stream) combination.

## 2.3 Load a pretrained SAE (Gemma Scope / SAELens)

We will load a pretrained SAE trained on **Gemma 2B** activations at our chosen layer.

In the Gemma Scope ecosystem, SAEs are often indexed by:
- model family (e.g., `gemma-scope-2b-pt-mlp-canonical`)
- layer (e.g., `layer_20`)
- width (e.g., `width_16k`)
- variant (e.g., `canonical`)

We will use a reasonable default that is commonly demonstrated.

If this exact identifier fails due to versioning differences, check the Gemma Scope repo or SAELens docs for the current naming.


In [ ]:
# ============================================================
# 2.3 Load SAE
# Note: this SAE was trained on the PT (base) model, not the IT model.
# Activations will have some distribution shift, but the feature space
# is similar enough for tutorial purposes. For research, use a matched SAE.
# ============================================================

SAE_RELEASE = cfg["sae_release"]
SAE_ID = cfg["sae_id"]

print("Loading SAE:", SAE_RELEASE, "|", SAE_ID)

result = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

sae = result[0] if isinstance(result, tuple) else result

print("SAE loaded.")
print("SAE d_in:", sae.cfg.d_in)
print("SAE n_features:", sae.cfg.d_sae)

Loading SAE: gpt2-small-res-jb | blocks.8.hook_resid_pre


SAE loaded.
SAE d_in: 768
SAE n_features: 24576


## 2.4 Activations → SAE → Sparse features → Reconstruction

We'll run a few prompts, cache activations at our hook point (`hook_resid_pre` for GPT-2, `hook_mlp_out` for Gemma), then:
- `x`: activation vectors from the model at the configured hook point
- `z`: sparse feature activations from the SAE encoder  
- `x_hat`: reconstructed activations from the SAE decoder


In [ ]:
# A small, varied prompt set
PROMPTS = [
    # Code
    "Write a short Python function that checks if a number is prime.",
    "Explain what this Python code does:\n\ndef foo(x):\n    return x**2",
    # Style
    "In Shakespearean English, write a love letter about rain.",
    "Write a short poem about the moon in Shakespearean English.",
    # Simple explanations
    "Explain photosynthesis like I'm 12 years old.",
    "Explain gravity like I'm 12 years old.",
    # Structured data
    """Here is a JSON object:
    {
    "name": "Ada",
    "role": "engineer"
    }
    Explain it.""",
    # Narrative
    "Two friends walk into a cafe and argue about whether AI can be creative.",
]


def get_hook_activations(prompts):
    """Return (tokens, acts) where acts has shape [batch, seq, d_model].
    The hook point depends on MODEL_FAMILY (residual stream for GPT-2,
    MLP out for Gemma).
    """
    tokens = model.to_tokens(prompts).to(device)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=lambda name: name == HOOK_POINT)
    acts = cache[HOOK_POINT]
    return tokens, acts


tokens, x = get_hook_activations(PROMPTS)
print("tokens shape:", tokens.shape)
print(f"x ({HOOK_POINT}) shape:", x.shape)

tokens shape: torch.Size([8, 46])
x (mlp_out) shape: torch.Size([8, 46, 768])


In [ ]:
# Flatten batch+seq so we can feed many vectors to the SAE at once
x_flat = x.reshape(-1, x.shape[-1])  # [batch*seq, d_model]
print("x_flat shape:", x_flat.shape)

with torch.no_grad():
    # Encode to sparse features
    z = sae.encode(x_flat)  # [batch*seq, n_features], sparse-ish
    # Decode back to activation space
    x_hat = sae.decode(z)  # [batch*seq, d_model]

print("z shape:", z.shape)
print("x_hat shape:", x_hat.shape)

# How sparse is z? (fraction of exactly-zero entries)
# Note: depending on implementation, you may see many ~0 rather than exact 0.
zero_frac = (z == 0).float().mean().item()
print("Fraction of exact zeros in z:", zero_frac)

x_flat shape: torch.Size([368, 768])
z shape: torch.Size([368, 24576])
x_hat shape: torch.Size([368, 768])
Fraction of exact zeros in z: 0.9985716342926025


## 2.5 Discover a few features via top-activating examples

A simple way to "name" features:

1. Pick a feature index \(k\).
2. Find the tokens / positions where \(z_{k}\) is largest.
3. Decode those tokens back to text context.

This is the basic workflow behind dashboards like Neuronpedia:
**feature → top activations → human labeling**.

Below we:
- compute top-activating feature indices overall
- show *a few* feature indices and their top-activating contexts

(For a real study you'd use thousands to millions of tokens; here we use a tiny set for speed.)


In [ ]:
# Aggregate feature activations across all content-token positions.
# We use MAX across positions: a feature is "active" in a prompt
# as long as it fires strongly at *any* position. This avoids diluting strong
# but localized activations (e.g., a feature that fires only on numbers will
# still rank highly even if most tokens don't trigger it).
#
# Alternative: use mean activation (smoother, less sensitive to single tokens).
# You can experiment by replacing .max(dim=0).values with .mean(dim=0).
z_np = z.detach().float().cpu()

# extract the token ID of BOS and PAD
bos_id = model.tokenizer.bos_token_id
pad_id = model.tokenizer.pad_token_id
if pad_id is None:
    pad_id = bos_id

# construct content mask: only retain the location of the real content token
tokens_flat = tokens.reshape(-1)  # [batch*seq]
content_mask = (tokens_flat != bos_id) & (tokens_flat != pad_id)  # [batch*seq] bool
content_mask_cpu = content_mask.cpu()

# Aggregate only at the content location — use max instead of mean
# The semantics of max: As long as there is a strong activation at any location, this feature is worth paying attention to.
z_content = z_np[content_mask_cpu]  # [n_content_tokens, n_features]
feature_strength = z_content.max(dim=0).values  # [n_features]

topk = 8
top_features = torch.topk(feature_strength, topk).indices.tolist()
print("Top feature indices (by max activation on content tokens):", top_features)

# At the same time, print the maximum activation value of each feature so you can get a sense of the magnitude
for f in top_features[:5]:
    print(f"  Feature {f}: max_act = {feature_strength[f]:.2f}")

Top feature indices (by max activation on content tokens): [9303, 21791, 17779, 9893, 11882, 13562, 14635, 13192]
  Feature 9303: max_act = 74.91
  Feature 21791: max_act = 65.23
  Feature 17779: max_act = 63.79
  Feature 9893: max_act = 58.39
  Feature 11882: max_act = 56.46


In [ ]:
def show_top_contexts_for_feature(feature_idx: int, k: int = 6):
    z_col = z_np[:, feature_idx].clone()

    bos_id = model.tokenizer.bos_token_id
    pad_id = model.tokenizer.pad_token_id if model.tokenizer.pad_token_id is not None else bos_id
    eos_id = model.tokenizer.eos_token_id if model.tokenizer.eos_token_id is not None else bos_id

    tokens_flat = tokens.reshape(-1)
    non_content = (tokens_flat == bos_id) | (tokens_flat == pad_id) | (tokens_flat == eos_id)
    z_col[non_content.cpu()] = 0.0

    # Take top-k only from positions with actual activations
    nonzero_mask = z_col > 0.0
    if nonzero_mask.sum() == 0:
        print(f"Feature {feature_idx} has no activations on the current prompt set.")
        return pd.DataFrame()
    effective_k = min(k, int(nonzero_mask.sum().item()))
    top_pos = torch.topk(z_col, effective_k).indices.tolist()

    _, seq_len = tokens.shape
    rows = []
    for flat_i in top_pos:
        b = flat_i // seq_len
        t = flat_i % seq_len
        tok_id = tokens[b, t].item()
        tok_str = model.to_string(tok_id)

        start = max(0, t - 12)
        end = min(seq_len, t + 12)
        ctx = model.to_string(tokens[b, start:end])

        rows.append(
            {
                "feature_idx": feature_idx,
                "batch": int(b),
                "pos": int(t),
                "token": repr(tok_str),
                "feature_act": float(z_col[flat_i]),
                "context_window": ctx.replace("\n", "\\n"),
            }
        )
    return pd.DataFrame(rows)


for f in top_features[:3]:
    print(f"\n=== Feature {f} ===")
    display(show_top_contexts_for_feature(f, k=6))


=== Feature 9303 ===



=== Feature 21791 ===



=== Feature 17779 ===


## 2.6 Make it visual with Neuronpedia (optional but recommended)

If the SAE release is integrated with Neuronpedia, you can inspect the same feature index on the dashboard.

Neuronpedia URLs often follow a pattern like:

- `https://neuronpedia.org/{MODEL}/{SAE}/{FEATURE}`

Different releases use different naming conventions.  
So instead of hard-coding a URL that might break, we provide a template and suggest checking the Gemma Scope / Neuronpedia docs for the exact mapping.

Example references:
- Neuronpedia homepage: https://neuronpedia.org/
- Gemma Scope (official, Google DeepMind): https://deepmind.google/models/gemma/gemma-scope/
- Gemma Scope on HuggingFace: https://huggingface.co/google/gemma-scope
- SAELens GitHub: https://github.com/decoderesearch/SAELens


In [ ]:
# ============================================================
# 2.7 Neuronpedia Visualization (requires internet connection)
# ============================================================
from IPython.display import IFrame
from IPython.display import display as ipy_display


NEURONPEDIA_BASE = "https://neuronpedia.org"
NP_MODEL_ID = cfg["np_model_id"]
NP_SAE_ID = cfg["np_sae_id"]


def check_and_show_neuronpedia_feature(feature_idx: int, height: int = 400):
    page_url = f"{NEURONPEDIA_BASE}/{NP_MODEL_ID}/{NP_SAE_ID}/{feature_idx}"
    embed_url = f"{page_url}?embed=true&embedStyle=minimal"

    api_url = f"{NEURONPEDIA_BASE}/api/feature/{NP_MODEL_ID}/{NP_SAE_ID}/{feature_idx}"
    try:
        resp = requests.get(api_url, timeout=5)
        if resp.status_code == 200:
            print(f"Feature {feature_idx}: {page_url}")
            ipy_display(IFrame(src=embed_url, width="100%", height=height))
        else:
            print(f"Feature {feature_idx} not indexed on Neuronpedia (HTTP {resp.status_code}).")
            print(f"You can still browse manually: {page_url}")
    except requests.exceptions.RequestException:
        print(f"  Network unavailable. Manual URL: {page_url}")


for f in top_features[:5]:
    check_and_show_neuronpedia_feature(f)

Feature 9303: https://neuronpedia.org/gpt2-small/8-res-jb/9303


Feature 21791: https://neuronpedia.org/gpt2-small/8-res-jb/21791


Feature 17779: https://neuronpedia.org/gpt2-small/8-res-jb/17779


Feature 9893: https://neuronpedia.org/gpt2-small/8-res-jb/9893


Feature 11882: https://neuronpedia.org/gpt2-small/8-res-jb/11882


# 3) Feature Steering Demo

Now the fun part: **control**.

High-level idea:
- A latent feature corresponds (approximately) to a direction in activation space given by the decoder row/column (implementation-dependent).
- If we add that direction at the right hook point, we can encourage the model to generate text associated with that feature.

We will:
1. Pick a feature that fires strongly in our mini prompt set.
2. Define a hook that adds `+α * direction` at `blocks.{LAYER}.hook_mlp_out`.
3. Compare baseline vs steered generation.

> Steering is not “proof of understanding,” but it is a useful causal probe.


In [ ]:
# Pick one feature to steer: use the strongest feature from our quick scan
STEER_FEATURE = top_features[0]
print("Steering feature index:", STEER_FEATURE)

# Decoder direction for that feature.
# In SAE implementations, W_dec is often shape [n_features, d_in]
# so W_dec[feature] is a vector in activation space (d_in).
direction = sae.W_dec[STEER_FEATURE].detach()

# Normalize so alpha is more interpretable
direction = direction / (direction.norm() + 1e-8)
print("direction norm:", direction.norm().item())

Steering feature index: 9303
direction norm: 1.0


In [ ]:
def generate_text(prompt: str, max_new_tokens: int = 60, temperature: float = 0.8, seed: int = 0):
    torch.manual_seed(seed)
    return model.generate(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.95,
        verbose=False,
    )


BASE_PROMPT = "Write a short paragraph about software engineering."
print("Baseline generation:")
print(generate_text(BASE_PROMPT, seed=1))

Baseline generation:
Write a short paragraph about software engineering. Start writing it, and write a word document. Then write a paragraph about software engineering.

Don't get too worked up. You can start with something you know from your job, or you can start with something you've been doing.

Still, here are some tips.


In [ ]:
# ============================================================
# 3.1 Diagnosis: what alpha magnitude should we use?
# ============================================================
sample_x = x_flat[content_mask_cpu][:20]
print(f"Content token activation norm (mean): {sample_x.norm(dim=-1).mean():.2f}")
print(f"Steering direction norm (unit): {direction.norm():.2f}")
print(
    "This is only a rough heuristic for choosing an initial alpha range. The effective steering strength depends on the model, layer, feature, and hook type."
)

Content token activation norm (mean): 111.27
Steering direction norm (unit): 1.00
This is only a rough heuristic for choosing an initial alpha range. The effective steering strength depends on the model, layer, feature, and hook type.


Important note:
In this demo we apply the steering direction to the entire activation tensor at the chosen hook point.
This is a global intervention across all token positions, not a position-specific causal edit.
We use it because it is simple and visually easy to demonstrate in a tutorial.

In [ ]:
# ============================================================
# 3.1 Steering hook: clamp up (positive alpha)
# ============================================================


def steering_hook(act, hook, alpha: float):
    # act: [batch, seq, d_model]
    return act + alpha * direction


def generate_with_steering(prompt: str, alpha: float, seed: int = 1, **gen_kwargs):
    torch.manual_seed(seed)
    np.random.seed(seed)

    fwd_hooks = [(HOOK_POINT, lambda act, hook: steering_hook(act, hook, alpha=alpha))]

    with model.hooks(fwd_hooks=fwd_hooks):
        return model.generate(
            prompt,
            max_new_tokens=gen_kwargs.get("max_new_tokens", 60),
            temperature=gen_kwargs.get("temperature", 0.8),
            do_sample=True,
            top_p=0.95,
            verbose=False,
        )


for alpha in [0.0, 10.0, 30.0, 60.0]:
    print("\n" + "=" * 70)
    print(f"Alpha = {alpha}")
    print(generate_with_steering(BASE_PROMPT, alpha=alpha, seed=1))


Alpha = 0.0
Write a short paragraph about software engineering. Start writing it, and write a word document. Then write a paragraph about software engineering.

Don't get too worked up. You can start with something you know from your job, or you can start with something you've been doing.

Still, here are some tips.

Alpha = 10.0
Write a short paragraph about software engineering. Start writing it, and write a word document. Then write a paragraph about software engineering.

Don't get me wrong. I know I'm not saying you can't do a lot of programming in your spare time, but I'm suggesting that, when you're done writing a paragraph about software

Alpha = 30.0
Write a short paragraph about software engineering. Start writing it, and write a word document. For example, I'll say that I'm an Amazon or B2B technical writer. After starting with this essay, I'll try to be precise about what I think you're going to write.  If you're being vague, I'll

Alpha = 60.0
Write a short paragraph abou

## 3.2 "Clamp to zero": Removing a feature from the SAE feature space

1. Extract the activation x at the hook point

2. Obtain the sparse code z using the SAE encoder, and force z[k] to 0

3. Reconstruct the modified activation using the SAE decoder

If a feature is causal to the model's behavior, its removal should result in a measurable change in the output.

> Note: this hook runs the full SAE encode → edit → decode path at every generation step, so it is computationally heavier than the additive steering used in Section 3.1. That is fine for a short tutorial demo, but it would become slow at larger scale.

In [ ]:
def generate_with_feature_clamped(prompt: str, feature_idx: int, seed: int = 1, **gen_kwargs):
    torch.manual_seed(seed)
    np.random.seed(seed)

    def hook_fn(act, hook):
        shape = act.shape  # [batch, seq, d_model]
        act_flat = act.reshape(-1, shape[-1])  # [batch*seq, d_model]

        with torch.no_grad():
            z_local = sae.encode(act_flat)  # [batch*seq, n_features]
            z_local[:, feature_idx] = 0.0  # Clamp to 0 in the feature space
            act_reconstructed = sae.decode(z_local)  # [batch*seq, d_model]

        return act_reconstructed.reshape(shape)

    fwd_hooks = [(HOOK_POINT, hook_fn)]

    with model.hooks(fwd_hooks=fwd_hooks):
        return model.generate(
            prompt,
            max_new_tokens=gen_kwargs.get("max_new_tokens", 60),
            temperature=gen_kwargs.get("temperature", 0.8),
            do_sample=True,
            top_p=0.95,
            verbose=False,
        )


print("Baseline:")
print(generate_text(BASE_PROMPT, seed=1))
print("\n" + "=" * 70)
print(f"Feature {STEER_FEATURE} clamped to zero:")
print(generate_with_feature_clamped(BASE_PROMPT, feature_idx=STEER_FEATURE, seed=1))

Baseline:
Write a short paragraph about software engineering. Start writing it, and write a word document. Then write a paragraph about software engineering.

Don't get too worked up. You can start with something you know from your job, or you can start with something you've been doing.

Still, here are some tips.

Feature 9303 clamped to zero:
Write a short paragraph about software engineering. Start writing your own article, especially about software development. Start writing a copy of your own writing. Write a copy of your technical information. After starting with a single paper, write a short article. Write your own article. Write a short article. Write a new article. Write a technical article.


# 4) The Dark Matter

Now we ask the uncomfortable question:

> Even if SAEs give us interpretable features, **how much of the model do they actually explain?**

There are two complementary ways to quantify “dark matter”:

1. **Activation-space fidelity** (local):  
   How well does $\hat{x}$ reconstruct $x$
   - Relative error: $$\|x-\hat{x}\|/\|x\|$$  
   - Variance explained: $$1 - \|x-\hat{x}\|^2 / \|x\|^2$$

2. **Behavioral fidelity** (downstream):  
   If we replace real activations with reconstructions, do the **logits / next-token distribution** stay similar?  
   A simple metric: KL divergence between original and reconstructed next-token distributions.

Below we compute both on our small prompt set.

> This is the “dark matter” punchline: interpretability tools usually explain some slice of computation, not the whole organism.


In [ ]:
# ============================================================
# 4.1 Activation-space dark matter (only compute content token)
# ============================================================

# Excluding BOS/PAD: their activation is approximately constant, and the SAE reconstruction quality is exceptionally high
# Including them in calculations would systematically overestimate their explanatory power
err = x_flat - x_hat  # [batch*seq, d_model]
err_c = err[content_mask_cpu]  # extract only the position of content
x_flat_c = x_flat[content_mask_cpu]

err_norm = err_c.norm(dim=-1)
x_norm = x_flat_c.norm(dim=-1) + 1e-8

rel_err = (err_norm / x_norm).mean().item()
explained = 1.0 - (err_c.pow(2).sum(dim=-1) / (x_flat_c.pow(2).sum(dim=-1) + 1e-8))
explained_mean = explained.mean().item()

print("Content tokens only (BOS/PAD excluded):")
print(f"Mean relative reconstruction error ||x-x̂||/||x||: {rel_err:.4f}")
print(f"Mean explained variance proxy: {explained_mean:.4f}")
print(
    f"Interpretable coverage is incomplete; about {(1 - explained_mean) * 100:.1f}% remains unexplained under this proxy."
)

plt.figure(figsize=(8, 4))
plt.hist(explained.detach().cpu().numpy(), bins=40, color="steelblue", edgecolor="white")
plt.axvline(explained_mean, color="red", linestyle="--", label=f"mean={explained_mean:.3f}")
plt.title("Explained Variance Distribution over Content Token Positions")
plt.xlabel("1 - ||x - x̂||² / ||x||²")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

Content tokens only (BOS/PAD excluded):
Mean relative reconstruction error ||x-x̂||/||x||: 0.3056
Mean explained variance proxy: 0.9006
Interpretable coverage is incomplete; about 9.9% remains unexplained under this proxy.


In [ ]:
# Pooled version of explained variance
num = err_c.pow(2).sum().item()
den = x_flat_c.pow(2).sum().item() + 1e-8
explained_pooled = 1.0 - num / den

print(f"Pooled explained variance: {explained_pooled:.4f}")
print(f"{(1 - explained_pooled) * 100:.1f}% of pooled activation energy remains unexplained.")

Pooled explained variance: 0.9040
9.6% of pooled activation energy remains unexplained.


## 4.2 Behavioral dark matter: do reconstructed activations preserve next-token logits?

We will do a simple intervention:

- Run the model normally on prompts → get next-token distribution at the final position.
- Run the model again but at `HOOK_POINT` replace MLP-out with the SAE reconstruction \(\hat{x}\).
- Compare next-token distributions via **KL divergence**.

This is not a perfect behavioral explanation metric, but it's a *useful sanity check:
If reconstruction is poor, logits can shift a lot. This metric only measures the final next-token distribution on the prompt. It does not fully capture generation-level or task-level behavioral fidelity.


In [ ]:
# ============================================================
# 4.2 Replace activations with reconstructions and measure KL
# ============================================================


def run_logits_with_reconstructed_activations(prompts: List[str]) -> torch.Tensor:
    toks = model.to_tokens(prompts).to(device)
    with torch.no_grad():
        # First run to get original activations at HOOK_POINT
        _, cache = model.run_with_cache(toks, names_filter=lambda name: name == HOOK_POINT)
        x0 = cache[HOOK_POINT]  # [batch, seq, d_model]
        x0_flat = x0.reshape(-1, x0.shape[-1])

        z0 = sae.encode(x0_flat)
        x0_hat_flat = sae.decode(z0)
        x0_hat = x0_hat_flat.reshape_as(x0)

    # Now run again, but overwrite HOOK_POINT with reconstructed values
    def overwrite_hook(act, hook):
        return x0_hat

    with torch.no_grad():
        return model.run_with_hooks(toks, fwd_hooks=[(HOOK_POINT, overwrite_hook)])


def next_token_probs_from_logits(logits: torch.Tensor) -> torch.Tensor:
    # logits: [batch, seq, vocab]
    final = logits[:, -1, :]
    return torch.softmax(final, dim=-1)


# Original logits
with torch.no_grad():
    toks = model.to_tokens(PROMPTS).to(device)
    orig_logits = model(toks)
orig_probs = next_token_probs_from_logits(orig_logits)

# Reconstructed-activation logits
recon_logits = run_logits_with_reconstructed_activations(PROMPTS)
recon_probs = next_token_probs_from_logits(recon_logits)

# KL(orig || recon) per prompt
kl = torch.nn.functional.kl_div(recon_probs.log(), orig_probs, reduction="none", log_target=False).sum(dim=-1)
print("KL(orig || recon) per prompt:")
display(pd.DataFrame({"prompt": PROMPTS, "KL": kl.detach().cpu().numpy()}))

print("Mean KL:", kl.mean().item())

## 4.3 Interpreting the numbers

- If **explained variance proxy** is high (close to 1), the SAE reconstruction is capturing most activation energy at that hook point.
- If **KL divergence** is low, replacing activations with reconstructions barely changes next-token behavior.
- If either number is poor, that is *literal* dark matter: the “feature dictionary” does not fully capture what the model is doing at that point.

Important nuance:
- This SAE covers **one layer** and **one stream** (`hook_mlp_out`).
- The model has many other streams and layers; the global dark matter is typically much larger than the local number you compute here.

### Why this matters for AI safety

If our interpretability method only explains a fraction of behavior, then:
- Safety-relevant computation may live in the unexplained remainder.
- Steering or monitoring a few features may miss crucial failure modes.

Interpretability is not magic X-ray vision; it’s a growing toolkit.  
Your job as a scientist is to **measure coverage** and stay honest about what remains unknown.


## 4.4 Summary (one-slide version)

### The four-act story of this tutorial

| Act | Central question | Tool used | What we found |
|-----|-----------------|-----------|---------------|
| **1 · Black box** | Why can't we just read the weights? | TransformerLens, model stats | Representations live in a $d_{\text{model}}$-dimensional space; individual neurons are **polysemantic** due to superposition — "one neuron = one concept" is a bad default |
| **2 · Decomposition** | Can we find a sparse, monosemantic basis? | Pretrained SAE (encoder + decoder) | Yes — the encoder maps each activation to a sparse code $z$ where most entries are zero; surviving features are often human-labelable |
| **3 · Causality** | Are features real causes, or just correlations? | Activation steering hooks | Adding or zeroing a decoder direction shifts model output in a semantically coherent way → features have **causal** leverage, not just correlational |
| **4 · Honest accounting** | How much do we actually explain? | Reconstruction error · KL divergence | Both metrics reveal a **non-trivial gap** at one hook point; the full model's "dark matter" across all layers is almost certainly larger |

---

### Key quantities to carry forward

$$
\underbrace{x}_{\text{activation at hook}} \;\xrightarrow{\;\text{SAE encoder}\;}\; \underbrace{z}_{\text{sparse code}} \;\xrightarrow{\;\text{SAE decoder}\;}\; \underbrace{\hat{x}}_{\text{reconstruction}}
$$

| Metric | Formula | What it measures |
|--------|---------|-----------------|
| Relative error | $$\|x - \hat{x}\| \;/\; \|x\|$$ | Local fidelity in activation space |
| Explained variance | $$1 - \|x - \hat{x}\|^2 \;/\; \|x\|^2$$ | Fraction of activation energy captured by the feature dictionary |
| Next-token KL | $$D_{\text{KL}}(p_{\text{orig}} \;\|\; p_{\text{recon}})$$ | Whether swapping $x \to \hat{x}$ changes what the model predicts next |

> **A single number to remember:** $1 - \text{explained variance}$ is the fraction of the model's local computation that our feature dictionary *cannot account for*. That is the dark matter.

---

### Take-home message

> SAEs give us **partial, local, and causal** access to LLM computation at one hook point.  
>  
> - *Partial* — reconstruction and behavioral dark matter are non-zero; the feature dictionary does not tell the whole story.  
> - *Local* — we tapped one layer, one stream; every other layer is still opaque.  
> - *Causal* — steering confirms that the features we recovered are not mere post-hoc labels; they have real influence on generation.  
>  
> Mechanistic interpretability is the project of **closing that dark matter gap** — layer by layer, circuit by circuit — until we can give a complete causal account of model behavior.



### Suggested extensions (beyond the bootcamp)

You now have a first-pass workflow for sparse feature analysis:
load a pretrained SAE, inspect feature activations, test simple interventions, and measure what the SAE fails to reconstruct.

Here are a few natural extensions:

1. **Layer sweep of dark matter.**
   Repeat the explained-variance and KL analysis across several layers. Which layer is most opaque to its SAE?

2. **Steering α sweep.**
   Sweep α for one strong feature and track the probability of a target token. Where does steering stop being coherent?

3. **Compare `max` vs. `mean` feature ranking.**
   Redo the feature-discovery step with `mean` instead of `max`. Which ranking feels more interpretable?

4. **SAE variant comparison.**
   Compare ReLU, TopK, or JumpReLU SAEs at the same hook point. Measure sparsity, fidelity, and interpretability.

5. **Minimal circuit tracing.**
   Pick a simple factual completion, find the layer where the answer token becomes likely, and inspect which SAE features are active there.

6. **Feature universality across model families.**
   Do similar semantic features appear in both GPT-2 and Gemma SAEs?

---

#### Pointers to go deeper

| Resource | What it gives you |
|----------|-------------------|
| [Anthropic's Scaling Monosemanticity (2024)](https://transformer-circuits.pub/2024/scaling-monosemanticity/) | Large-scale SAE feature catalogs on Claude 3 Sonnet; the current state of the art |
| [Neuronpedia](https://neuronpedia.org/) | Interactive dashboards for GPT-2, Gemma-2, Llama features; start exploring without writing any code |
| [SAELens](https://github.com/jbloomAus/SAELens) | The library powering this tutorial; has training scripts and a model hub |
| [TransformerLens](https://github.com/neelnanda-io/TransformerLens) | Activation caching, patching, and circuit analysis for any HuggingFace-compatible transformer |
| [ARENA Mech Interp curriculum](https://github.com/callummcdougall/ARENA_3.0) | Structured exercises that build directly on what you've done here |
| [Open Problems in Mechanistic Interpretability (2025)](https://arxiv.org/pdf/2501.16496)| A forward-looking map of the field: open questions in methods, applications, and foundations, useful for finding concrete project directions beyond toy demos
